# Battery Degradation Model Training
This notebook loads real battery data and trains the XGBoost degradation predictor.

In [37]:
import sys
sys.path.append('..')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from src.degradation.predictor import DegradationPredictor

# Load the real dataset
df = pd.read_csv('../data/raw/Battery_dataset.csv')
print('Loaded', len(df), 'rows')
print(df.head())

ImportError: cannot import name 'PANDAS_INSTALLED' from 'xgboost.compat' (/Users/tameeztayob/Downloads/bms-ai/venv/lib/python3.12/site-packages/xgboost/compat.py)

In [ ]:
# Plot SoH over cycles for each battery
for battery in df['battery_id'].unique():
    subset = df[df['battery_id'] == battery]
    plt.plot(subset['cycle'], subset['SOH'], label=battery)

plt.axhline(70, color='red', linestyle='--', label='End-of-Life (70%)')
plt.xlabel('Cycle')
plt.ylabel('State of Health (%)')
plt.title('Real Battery Degradation')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Prepare features and target for training
features = ['cycle', 'chI', 'chV', 'chT', 'disI', 'disV', 'disT', 'BCt']
target = 'SOH'

X = df[features]
y = df[target]

print('Features shape:', X.shape)
print('Target range: {:.1f}% to {:.1f}%'.format(y.min(), y.max()))

In [ ]:
# Train the AI model
predictor = DegradationPredictor(model_path='../models/degradation_model.pkl')
metrics = predictor.train(X, y)

print('\n--- Model Results ---')
print(f'Mean Absolute Error: {metrics["mae"]:.3f}%')
print(f'R² Score:            {metrics["r2"]:.4f}  (1.0 = perfect)')

In [ ]:
# Save the trained model
predictor.save()
print('Model saved to models/degradation_model.pkl')

In [ ]:
# Plot predicted vs actual SoH
predictions = predictor.predict(X)

plt.figure(figsize=(10, 4))
plt.plot(df['cycle'][:220], y[:220], label='Actual SoH', alpha=0.7)
plt.plot(df['cycle'][:220], predictions[:220], label='Predicted SoH', linestyle='--', alpha=0.7)
plt.xlabel('Cycle')
plt.ylabel('SoH (%)')
plt.title('AI Prediction vs Real Data (Battery B5)')
plt.legend()
plt.tight_layout()
plt.show()